# Exercise 4.1: Your First LangGraph — From Chains to Graphs

**Module:** 4 — LangGraph
**Level:** Intermediate

Chains are linear: A → B → C. But real workflows need **decisions** and **branches**. That's where **LangGraph** comes in — it lets you build workflows as **graphs** with nodes and edges.

**What you'll do:**
1. Understand the three building blocks: State, Nodes, Edges
2. Build a two-step graph: research → summarize
3. Visualize the graph structure
4. Trace execution to see data flow through the graph

**Prerequisite:** Complete 3.1 (Your First Chain) first.

## 1. Setup

Install LangGraph alongside LangChain.

Get your free Groq API key at: https://console.groq.com/keys

In [ ]:
# Install LangGraph (the graph framework) and LangChain with Groq
# langgraph is built on top of langchain — they work together
!pip install langgraph langchain langchain-groq -q

In [ ]:
import os

# Paste your Groq API key between the quotes
# Get it free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. Chains vs Graphs — Why Upgrade?

**Chains** are great for linear workflows:
```
prompt → model → parser  (always the same path)
```

**Graphs** let you build workflows with:
- **Branching:** "If the disruption is severe, escalate. Otherwise, auto-resolve."
- **Looping:** "Keep retrying until quality is good enough."
- **Parallel paths:** "Research flights AND hotels at the same time."

```
Chain:   A → B → C
Graph:   A → B → (if X: C, if Y: D) → E
```

## 3. The Three Building Blocks

Every LangGraph has exactly three things:

### State
A dictionary that **flows through the graph**. Every node can read and update it. Think of it as a shared clipboard that gets passed from step to step.

### Nodes
Functions that **do work**. Each takes the current state, does something (call an LLM, process data, make a decision), and returns updates to the state.

### Edges
**Connections** between nodes. Data flows along edges. You define the order: "after node A, go to node B."

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# === STEP 1: Define the State ===
# State is a TypedDict — a dictionary with defined keys and types.
# Every node receives this state and can update any of its fields.


class FlightState(TypedDict):
    flight_info: str       # Input: raw flight disruption info
    research: str          # Filled by the research node
    summary: str           # Filled by the summarize node


# When the graph starts, only flight_info will be filled.
# Each node will fill its own field as data flows through.
print("State defined with 3 fields: flight_info → research → summary")
print("Think of it as a form that gets filled out step by step.")

In [ ]:
# === STEP 2: Define the Nodes ===
# Each node is a function that takes state and returns a dict of updates.

# Create the LLM we'll use in the nodes
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


def research_node(state: FlightState) -> dict:
    """Node 1: Research the flight disruption.
    Takes flight_info from state, produces research."""

    # Read the input from state
    flight_info = state["flight_info"]

    # Call the LLM to analyze the disruption
    response = model.invoke(
        f"""You are a flight operations analyst. Analyze this disruption:

{flight_info}

Provide:
1. Severity assessment (minor/moderate/severe)
2. Affected passengers estimate
3. Root cause analysis
4. Two recommended actions"""
    )

    # Return updates to state — only the fields we want to change
    # The graph merges this with the existing state
    return {"research": response.content}


def summarize_node(state: FlightState) -> dict:
    """Node 2: Summarize the research into a passenger notification.
    Takes research from state, produces summary."""

    # Read the research that the previous node produced
    research = state["research"]

    # Call the LLM to create a passenger-friendly summary
    response = model.invoke(
        f"""Based on this analysis, write a clear, empathetic passenger notification.
Keep it under 4 sentences. Include any alternative arrangements.

Analysis:
{research}"""
    )

    return {"summary": response.content}


print("Two nodes defined:")
print("  1. research_node: analyzes the disruption")
print("  2. summarize_node: creates passenger notification")

In [ ]:
# === STEP 3: Build the Graph ===
# Connect nodes with edges to define the execution order.

# Create a new graph with our state type
graph_builder = StateGraph(FlightState)

# Add nodes — give each a name and the function to call
graph_builder.add_node("research", research_node)
graph_builder.add_node("summarize", summarize_node)

# Add edges — define the flow order
# START is a special node meaning "entry point"
# END is a special node meaning "we're done"
graph_builder.add_edge(START, "research")       # Start → research
graph_builder.add_edge("research", "summarize")  # research → summarize
graph_builder.add_edge("summarize", END)          # summarize → done

# Compile the graph — this validates the structure and creates a runnable
graph = graph_builder.compile()

print("Graph compiled!")
print("Flow: START → research → summarize → END")

## 4. Visualize the Graph

LangGraph can generate a visual diagram of your graph. This is incredibly useful for debugging complex workflows.

In [ ]:
from IPython.display import Image, display

# get_graph() returns the graph structure, draw_mermaid_png() renders it as an image.
# This uses the Mermaid diagram format — a popular text-to-diagram tool.
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # If the rendering fails (missing dependencies), show the text version
    print("Graph structure (text):")
    graph.get_graph().print_ascii()
    print(f"\n(For the visual diagram, install: pip install pygraphviz)")

## 5. Run the Graph

Let's feed a flight disruption into the graph and watch it flow through both nodes.

In [ ]:
# Define the initial state — only fill in the input field
initial_state = {
    "flight_info": """Flight TK1234 from Istanbul (IST) to Paris (CDG):
- Scheduled departure: 09:30
- Current status: Delayed 3 hours (mechanical issue)
- 180 passengers booked
- 45 passengers have connecting flights at CDG"""
}

# Run the graph — invoke() starts at START and follows edges until END
# The graph fills in research and summary as it goes
result = graph.invoke(initial_state)

# result is the final state with ALL fields filled
print("=" * 60)
print("FINAL STATE")
print("=" * 60)
print(f"\n--- Input ---")
print(result["flight_info"])
print(f"\n--- Research (Node 1 output) ---")
print(result["research"])
print(f"\n--- Summary (Node 2 output) ---")
print(result["summary"])

## 6. Tracing Execution Step by Step

You can use `.stream()` instead of `.invoke()` to see each node's output as it happens. This is great for debugging.

In [ ]:
# stream() yields the state updates after each node completes.
# This lets you see the data flowing through the graph in real time.

another_disruption = {
    "flight_info": """Flight TK5678 from Istanbul (IST) to London (LHR):
- Scheduled departure: 14:00
- Current status: Cancelled (severe weather at LHR)
- 210 passengers booked
- Next available flight: tomorrow 08:00"""
}

print("Streaming graph execution...\n")

# Each step yields {node_name: {state_updates}}
for step in graph.stream(another_disruption):
    # step is a dict with one key: the node that just completed
    node_name = list(step.keys())[0]
    print(f">>> Node '{node_name}' completed <<<")
    # Show a preview of what the node produced
    for key, value in step[node_name].items():
        preview = value[:200] + "..." if len(value) > 200 else value
        print(f"  {key}: {preview}")
    print()

## 7. Why Graphs Over Chains?

Right now, our graph is linear (research → summarize) — basically a chain. So why bother?

Because graphs unlock **conditional routing**: "If the disruption is severe, add an escalation step." We'll build that in the next notebook (4.2).

For now, the key insight is:
- **State** is a shared data container
- **Nodes** read from and write to state
- **Edges** define the execution order
- The graph **manages the flow** automatically

In [ ]:
# Quick comparison: the same workflow as a chain vs a graph

# CHAIN (from Module 3):
# chain = prompt1 | model | parser | prompt2 | model | parser
# Simple, linear, no branching possible.

# GRAPH (what we just built):
# graph: START → research → summarize → END
# Same flow now, but ready for conditional edges, loops, parallelism.

print("Chain:  prompt → model → parser  (fixed path)")
print("Graph:  START → [nodes with edges] → END  (flexible path)")
print("\nKey difference: graphs can BRANCH based on data in the state.")
print("That's what we'll build in Exercise 4.2.")

---

## YOUR TURN: Add a Third Node

Extend the graph with a **translate** node that takes the summary and translates it to Turkish (or any language you choose).

The flow should be: `START → research → summarize → translate → END`

Hints:
1. Add a `translation` field to the State
2. Create a `translate_node` function that reads `summary` and returns `translation`
3. Add the node and update the edges

In [ ]:
# YOUR CODE HERE

# 1. Define extended state
# class ExtendedFlightState(TypedDict):
#     flight_info: str
#     research: str
#     summary: str
#     translation: str    # <-- New field

# 2. Define translate_node
# def translate_node(state) -> dict:
#     ...
#     return {"translation": response.content}

# 3. Build the graph
# graph_builder = StateGraph(ExtendedFlightState)
# graph_builder.add_node(...)
# graph_builder.add_edge(...)
# graph = graph_builder.compile()

# 4. Run and print the translation



## Key Takeaways

- **State** (`TypedDict`) is a dictionary that flows through the graph — shared data
- **Nodes** are functions: take state, do work, return updates
- **Edges** connect nodes: define the execution order
- **`graph.invoke()`** runs the full graph; **`graph.stream()`** shows step-by-step
- **`get_graph().draw_mermaid_png()`** visualizes the graph structure

**Next:** In Exercise 4.2, you'll add **conditional routing** — graphs that make decisions based on the data.